# K513 · Week 6, Session 1
## When accuracy is not the answer

Last Thursday you built a decision tree and used two accuracy scores to decide it was too deep. Today you meet a dataset where the accuracy score is high, stable, and almost meaningless — and you learn what to look at instead.

No new algorithm today. The same logistic regression and the same decision tree you already know, read four different ways.

---
### Before you type anything

**File → Save a copy in Drive.**

This notebook is read-only for you. You can type into it and run it and it will look completely
normal, but nothing you do will be saved. Save your own copy first, every time.

---

### Using AI in this notebook

Gemini is built into Colab and you are welcome to use it here. Two things worth knowing:

- It does not know which columns you have or what we covered in class. Whatever it writes, you own.
- The most useful thing you can ask it is **"explain what this line does"** — not "write it for me".


One more, specific to today: an AI will happily report a model's accuracy and tell you it looks good. It cannot see that only 9.6% of your rows are positives, because that is a fact about your data rather than about your code. The judgment on this page is yours.

---

### Turn off Unwanted AI Assistance

AI-powered coding completion is turned on by default. It is convenient but does not give you a chance
to think and learn. Turning it off helps you learn. You can always turn it back on when needed.
- Tools → settings → AI Assistance → Uncheck "Show AI-powered inline code completions"
- Tools → settings → Uncheck "Show context-powered code completions"

---

### How to run a cell

Click on a cell, then press **Shift + Enter**. That runs it and moves you to the next one.

---

## 1. Set up

Run the next two cells. Nothing to fill in.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (confusion_matrix, accuracy_score, recall_score,
                             precision_score, f1_score, classification_report)

LOAN_URL = "https://raw.githubusercontent.com/jl-uscn/k513-data/main/loan.csv"
SEED = 42

### The data

Five thousand customers from the bank's last personal loan campaign.

| Column | |
|---|---|
| `Income` | annual income, in thousands of dollars |
| `CDAccount` | 1 if the customer holds a certificate of deposit with the bank, 0 if not |
| `PersonalLoan` | **the target.** 1 if the customer accepted the loan offer, 0 if not |

The file has more columns than these. We use two on purpose — a deliberately weak model is the
only way to watch one measure of quality point at a different model than another one does.

In [ ]:
loan_df = pd.read_csv(LOAN_URL)
print("Rows and columns:", loan_df.shape)
loan_df[['Income', 'CDAccount', 'PersonalLoan']].head()

### How rare is the thing we are trying to find?

In [ ]:
print(loan_df['PersonalLoan'].value_counts())
print()
print(loan_df['PersonalLoan'].value_counts(normalize=True).round(4))

**Look at that second number before you go on.** Fewer than one customer in ten accepted.
That single fact is what makes everything below behave the way it does.

---

## 2. Build the three models

Nothing here is new. Same split, same pipeline, same two model classes as last week — plus the
baseline, which predicts the commoner class and nothing else.

In [ ]:
X = loan_df[['Income', 'CDAccount']]
y = loan_df['PersonalLoan']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y)

print(f"Training: {len(X_train)} customers, {y_train.sum()} of them acceptors")
print(f"Test:     {len(X_test)} customers, {y_test.sum()} of them acceptors")

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), ['Income']),
    ('bin', 'passthrough', ['CDAccount'])])

baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)

logreg = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=SEED))]).fit(X_train, y_train)

tree = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(X_train, y_train)

print("three models fitted")

### Their accuracy on the test customers

In [ ]:
for name, model in [('Baseline      ', baseline),
                    ('Logistic      ', logreg),
                    ('Decision tree ', tree)]:
    print(f"{name} {model.score(X_test, y_test):.4f}")

Three numbers within one and a half points of each other, and one of them belongs to a
model that does not look at the data at all.

---

## ✏️ Now You Try · 1 — build all three tables

**About 12 minutes.** Work with a neighbor.

A confusion matrix is a cross-tabulation of what really happened against what the model said —
the same shape as an Excel pivot table with one field in Rows and one in Columns.

Both of the calls below take **what really happened as the first argument** and **what the model said as the second**.
Get them the wrong way round and you get the transpose, which looks perfectly reasonable and is
not.

### (a) The pivot table you already know

Fill in the two blanks. `y_test` is the actual target value (what really happened); `y_pred_logreg` is what the model said.

In [ ]:
y_pred_baseline = baseline.predict(X_test)
y_pred_logreg   = logreg.predict(X_test)
y_pred_tree     = tree.predict(X_test)

pd.crosstab(index=____, columns=____,
            rownames=['Actually'], colnames=['Model said'],
            margins=True, margins_name='Total')

### (b) The same table, from scikit-learn

`confusion_matrix()` returns the four counts as an array, laid out like this:

```
[[ customers who said no,  and the model agreed        customers who said no,  and the model said accept ],
 [ customers who accepted, and the model said no       customers who accepted, and the model agreed      ]]
```

Build one for each of the three models.

In [ ]:
cm_baseline = confusion_matrix(y_test, y_pred_baseline)
cm_logreg   = ____
cm_tree     = ____

print("Baseline\n", cm_baseline)
print("\nLogistic regression\n", cm_logreg)
print("\nDecision tree\n", cm_tree)

### (c) Say the four numbers out loud, in customers

Take the logistic regression matrix. Say each of its four numbers to your neighbor as a sentence
about **customers** — not as "true positive" or "false negative".

The helper below unpacks them for you. Read its output; there is nothing to fill in.

In [ ]:
tn, fp, fn, tp = cm_logreg.ravel()
print(f"{tn:>5} customers would have said no,  and the model left them alone")
print(f"{fp:>5} customers would have said no,  and we mailed them anyway")
print(f"{fn:>5} customers would have accepted, and we never contacted them")
print(f"{tp:>5} customers would have accepted, and we found them")

### (d) How many of the acceptors did each model find?

There were 120 acceptors among the test customers. Write the three numbers on your card.

In [ ]:
for name, cm in [('Baseline     ', cm_baseline),
                 ('Logistic     ', cm_logreg),
                 ('Decision tree', cm_tree)]:
    found = cm[1, 1]
    print(f"{name} found {found:>3} of the 120")

### (e) One sentence

Which model would you now put in front of the marketing department, and what changed your mind?

> ✏️ **Not collected.** This one is for the room — be ready to say it out loud.

---

## ✏️ Now You Try · 2 — and then recommend one

**About 10 minutes.**

Two measures, both built from the four counts you already have.

| | |
|---|---|
| **recall** | acceptors found ÷ acceptors there were |
| **precision** | acceptors found ÷ customers you mailed |

Same numerator. Different denominator. Recall divides by what was out there; precision divides by
what you went after.

### (a) Compute both, for both real models

Fill in the two blanks. Each function takes what really happened first.

In [ ]:
for name, y_pred in [('Logistic     ', y_pred_logreg),
                     ('Decision tree', y_pred_tree)]:
    r = ____(y_test, y_pred)
    p = ____(y_test, y_pred)
    print(f"{name}  recall {r:.3f}   precision {p:.3f}")

Check them against your own confusion matrices. Recall for the logistic model is
`tp / (tp + fn)` — the bottom row of the table. Precision is `tp / (tp + fp)` — the right-hand
column.

In [ ]:
tn, fp, fn, tp = cm_logreg.ravel()
print(f"recall    from the table: {tp} / ({tp} + {fn}) = {tp / (tp + fn):.3f}")
print(f"precision from the table: {tp} / ({tp} + {fp}) = {tp / (tp + fp):.3f}")

### (b) The whole report in one call

Find your two numbers in the output. Read the **Accepted (1)** row and ignore almost everything
else — `support` is how many of that class were really there.

In [ ]:
print(classification_report(y_test, y_pred_logreg, digits=3,
                            target_names=['Said no (0)', 'Accepted (1)']))

### (c) What does each model earn?

The bank's own numbers: an accepted loan is worth about **\$500**, and every mailing costs
**\$10** whether the customer accepts it or not.

So for any model:

> **profit  =  \$500 × acceptors reached  −  \$10 × customers mailed**

Fill in the two blanks. **Include the baseline** — it mails nobody, so watch what it earns.

In [ ]:
for name, cm in [('Baseline     ', cm_baseline),
                 ('Logistic     ', cm_logreg),
                 ('Decision tree', cm_tree)]:
    tn, fp, fn, tp = cm.ravel()
    mailed = ____
    profit = 500 * ____ - 10 * mailed
    print(f"{name}  mailed {mailed:>4}, {tp:>3} accepted  ->  profit ${profit:>7,}")

### (d) The baseline earned \$0 — and spent \$0

It mailed nobody, so it wasted nothing and risked nothing. On a pure cash view it is the safest
thing the bank could do.

Run the cell below, then argue it out with your neighbor.

In [ ]:
BUDGET       = 1250   # dollars approved for the campaign, on our 1,250 customers
PER_MAILING  = 10     # what one mailing costs
PER_ACCEPTOR = 500    # what one accepted loan is worth

def show(label, acceptors_reached, mailed):
    cost = PER_MAILING * mailed
    profit = PER_ACCEPTOR * acceptors_reached - cost
    over = "   over budget" if cost > BUDGET else ""
    print(f"{label:<34}{mailed:>9}{cost:>10,}{profit:>11,}{over}")

print(f"{'What the bank could do':<34}{'mailings':>9}{'cost':>10}{'profit':>11}")
show('Mail nobody - the baseline', 0, 0)
for name, cm in [('Decision tree, default cut', cm_tree),
                 ('Logistic regression, default cut', cm_logreg)]:
    tn, fp, fn, tp = cm.ravel()
    show(name, tp, tp + fp)
show('Mail everyone', 120, 1250)
show('A perfect model - only the 120', 120, 120)

print(f"\nThe campaign budget is ${BUDGET:,}.")

### (e) Three questions to argue about

1. What does the bank **give up** by doing nothing? Is that a cost, exactly?
2. Mailing everyone earns more than any model on that list — and costs **\$12,500** against a
   **\$1,250** budget. Suppose the money were there. Would you still do it?
3. In one sentence to the head of marketing: which model would you run, and why is running
   nothing the wrong answer?

> ✏️ **Not collected.** Be ready to argue it.

### Before you answer question 2

Every number in that table was computed on 1,250 customers whose answers we **already know**. None
of it is a measurement of next year.

The 9.6% came from one campaign, to 5,000 customers who were chosen somehow. The \$500 is an
estimate of what a loan is worth over its life. Break-even is a **2%** response rate — so the whole
case for mailing everyone rests on next year's rate staying above 2%, and nobody has tested that.

Mailing everyone bets **\$12,500** on that assumption. The model bets **\$730** on the same
assumption. That difference is most of what a finance department is arguing about when it says no.

This is worth carrying past this course: a model does not tell you the future. It tells you who
looks most like the people who said yes last time. **The response rate is fragile; the ordering is
more robust** — and that is both why the model is worth having and why it does not settle the
argument on its own.

---

## 3. Your friend's test

A screening test that is right 90% of the time. One person in a hundred truly has the disease.
Imagine 100,000 people take it.

**Same layout as the three confusion matrices you built in section 1** — the negative class first,
the positive class second, so the people who really have the disease and were caught sit in the
bottom-right corner.

Run the cell and read the table before you read the number underneath it.

In [ ]:
N, accuracy, prevalence = 100_000, 0.90, 0.01

have_it     = round(prevalence * N)
not_have_it = N - have_it

true_positives  = round(accuracy * have_it)        # has it, and the test says so
false_negatives = have_it - true_positives         # has it, and the test misses it
false_positives = round((1 - accuracy) * not_have_it)   # does not have it, flagged anyway
true_negatives  = not_have_it - false_positives

print(pd.DataFrame(
    {'Test says negative': [true_negatives, false_negatives],
     'Test says positive': [false_positives, true_positives]},
    index=[f'{not_have_it:,} who do not have it', f'{have_it:,} who have it']))

flagged = false_positives + true_positives
print(f"\nPositive results in total: {flagged:,}")
print(f"Of those, the share who really have it: {true_positives / flagged:.1%}")

**That is precision.** Of everyone the test flagged, the share who really have it. Your
friend was told the *accuracy*.

Now change one thing — how common the disease is — and watch the answer move without the test
changing at all.

In [ ]:
for p in [0.001, 0.01, 0.05, 0.10, 0.30]:
    tp = accuracy * p * N
    fp = (1 - accuracy) * (N - p * N)
    print(f"If {p:>6.1%} of people have it: "
          f"a positive result means a {tp / (tp + fp):>6.1%} chance")

The number she needed was never a fact about the test.

This is also why screening is offered to groups who are already at higher risk: move down that
list and the same test tells you far more.

---

## Before you close this notebook

You now have four ways to read a classifier and one rule for choosing between them:

| | |
|---|---|
| **accuracy** | fine when the classes are balanced and both mistakes cost the same |
| **recall** | when missing a real case is what hurts |
| **precision** | when acting on a false alarm is what hurts |
| **F1** | when nobody will tell you what the mistakes cost |

On Thursday: the same model, at every possible setting, and how to choose one.

**The weekly homework covers both sessions** and is posted on Canvas as
*Week 6 Homework - Model Evaluation*.